# Chestband Monitor - Notebook

This notebook reuses `simulate.py` and `analyze.py` directly, so the notebook and the
standalone scripts can never drift out of sync. It:

- Generates the full 30-day dataset in memory in one pass and writes the CSV **once**.
- Shows the full 30-day trend and text report.
- Runs a bounded, fast "live playback" preview at the end that samples the already-computed
  data (no re-simulation, no repeated disk writes), so it finishes in seconds.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Make sure Python can find simulate.py / analyze.py when they live next to this notebook.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import simulate as sim
import analyze as analyze

print(f"Data will be read/written at: {sim.OUTPUT_FILE}")

In [ ]:
# Generate the dataset once (in memory), then write the CSV ONCE.
if sim.OUTPUT_FILE.exists():
    print("Existing dataset found, loading it. Delete the file (or call sim.create_dataset() yourself) to regenerate.")
    df = pd.read_csv(sim.OUTPUT_FILE)
else:
    print("No existing dataset found - simulating 30 days now (this takes a little while)...")
    df = sim.create_dataset()
    df.to_csv(sim.OUTPUT_FILE, index=False)
    print(f"Saved {len(df):,} rows to {sim.OUTPUT_FILE}")

df.head()

In [ ]:
# Reuse the same analysis the standalone script produces, so the notebook and the
# script never disagree.
analyze.main()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df["simulation_day"], df["breathing_rate_bpm"], linewidth=0.7)
plt.axhline(12, linestyle="--", color="orange", label="Low threshold: 12/min")
plt.axhline(20, linestyle="--", color="red", label="High threshold: 20/min")
plt.xlim(0, 30)
plt.xlabel("Simulation day")
plt.ylabel("Breathing frequency [breaths/min]")
plt.title("Breathing Frequency Over 30 Days")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Bounded live-playback preview

This replays the simulation at high speed for a quick visual sanity check. It never
re-simulates or re-writes the CSV per frame, tracks stats incrementally, and only
redraws every `FRAME_STRIDE` minutes up to `MAX_FRAMES` redraws total - so it finishes
in seconds instead of looping over all 43,200 simulated minutes.

In [ ]:
SUSTAINED_HIGH_MINUTES = 10
FRAME_STRIDE = 60          # advance this many simulated minutes per redraw
MAX_FRAMES = 200           # cap total redraws so this always finishes quickly
FRAME_DELAY_SECONDS = 0.03

total_rows = len(df)
frame_indices = list(range(FRAME_STRIDE - 1, total_rows, max(FRAME_STRIDE, total_rows // MAX_FRAMES)))

br_values = df["breathing_rate_bpm"].to_numpy()
status_values = df["status"].to_numpy()
day_values = df["simulation_day"].to_numpy()

for frame_num, end_idx in enumerate(frame_indices):
    chunk_br = br_values[: end_idx + 1]
    chunk_status = status_values[: end_idx + 1]

    run = 0
    episodes = 0
    for s in chunk_status:
        if s == "High":
            run += 1
            if run == SUSTAINED_HIGH_MINUTES:
                episodes += 1
        else:
            run = 0

    clear_output(wait=True)
    print("Chestband monitor (bounded preview)")
    print(f"Simulation day: {day_values[end_idx]:.2f} / 30")
    print(f"Breathing rate: {chunk_br[-1]:.1f} breaths/min")
    print(f"Status: {chunk_status[-1]}")
    print(f"Average so far: {chunk_br.mean():.2f} /min")
    print(f"Sustained high episodes so far: {episodes}")
    print(f"Frame {frame_num + 1} / {len(frame_indices)}")

    plt.figure(figsize=(10, 3))
    plt.plot(day_values[: end_idx + 1], chunk_br, linewidth=0.8)
    plt.axhline(12, linestyle="--", color="orange")
    plt.axhline(20, linestyle="--", color="red")
    plt.xlim(0, 30)
    plt.xlabel("Simulation day")
    plt.ylabel("Breaths/min")
    plt.title("Breathing Frequency (live preview)")
    plt.grid(True, alpha=0.3)
    plt.show()

    time.sleep(FRAME_DELAY_SECONDS)

print("Preview complete.")